In [1]:
import os
import json
import random
import pathlib
import pickle
import warnings
from collections import Counter

import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
BASE_DIR = pathlib.Path(os.getcwd()).resolve().parents[1] 
DATA_ROOT = BASE_DIR / "data" / "cats_dogs"
TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"
PROCESSED_ROOT = BASE_DIR / "experiments" / "cat-dog_voice" / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = PROCESSED_ROOT / "data_manifest.pkl"
AUDIT_PATH = BASE_DIR / "experiments" / "cat-dog_voice" / "results" / "dataset_audit.json"
AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [4]:
def wav_to_logmel(wav_path: pathlib.Path, sr_target: int = 22050, duration: float = 3.0) -> np.ndarray:
    """Return a (64, 64) log‑Mel spectrogram (single channel)."""
    # Load raw audio (librosa handles many formats)
    y, sr = sf.read(wav_path)
    if y.ndim > 1:
        y = np.mean(y, axis=1)  # mono
    # Resample if needed
    if sr != sr_target:
        y = librosa.resample(y, orig_sr=sr, target_sr=sr_target)
        sr = sr_target
    # Pad / truncate to fixed length
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")
    else:
        y = y[:target_len]
    # Compute Mel spectrogram
    S = librosa.feature.melspectrogram(
        y,
        sr=sr,
        n_fft=1024,
        hop_length=512,
        n_mels=64,
        fmax=sr // 2,
    )
    log_S = librosa.power_to_db(S, ref=np.max)
    # Normalize to [0, 1]
    log_S_norm = (log_S - log_S.min()) / (log_S.max() - log_S.min() + 1e-6)
    return log_S_norm.astype(np.float32)

In [5]:
print("🔎 Auditing raw dataset …")
class_counts = Counter()
all_files = []
for class_dir in ["cat", "dog"]:
    class_path = TRAIN_ROOT / class_dir
    for wav in class_path.rglob("*.wav"):
        all_files.append((wav, class_dir))
        class_counts[class_dir] += 1
print(f"Found {len(all_files)} training wavs – class distribution: {dict(class_counts)}")
# Simple corruption check – try loading first 5 seconds of each file
corrupt = []
for wav, _ in all_files:
    try:
        _ = sf.read(wav, frames=22050 * 1)  # 1 s preview
    except Exception:
        corrupt.append(str(wav))
if corrupt:
    print(f"⚠️ Detected {len(corrupt)} corrupt files – they will be skipped.")
else:
    print("✅ No corrupt files detected.")

🔎 Auditing raw dataset …
Found 0 training wavs – class distribution: {}
✅ No corrupt files detected.
